In [10]:
import pandas as pd
import time
from newspaper import Article
from concurrent.futures import ThreadPoolExecutor, as_completed
from googlenewsdecoder import new_decoderv1


df_all = pd.read_csv("Kelompok 5 - Link Artikel LPDP - All.csv")
df_valid = df_all[df_all["Valid?"] == True].copy().reset_index(drop=True)

print(f"Total artikel valid: {len(df_valid)}")

if "Actual_URL" not in df_valid.columns:
    df_valid["Actual_URL"] = None
if "Content" not in df_valid.columns:
    df_valid["Content"] = None


def safe_decode_url(args):
    idx, url = args
    for attempt in range(2):
        try:
            result = new_decoderv1(url)
            if result.get("status"):
                return idx, result["decoded_url"]
        except Exception as e:
            if attempt == 0:
                time.sleep(0.5)
    return idx, None


def scrape_content(args):
    idx, url = args
    if url is None or pd.isna(url):
        return idx, None
    try:
        article = Article(url, language="id", request_timeout=8)
        article.download()
        article.parse()
        text = article.text.strip()
        return idx, text if len(text) > 50 else None
    except Exception:
        return idx, None


need_decode = [
    (i, df_valid.loc[i, "URL"])
    for i in range(len(df_valid))
    if pd.isna(df_valid.loc[i, "Actual_URL"])
]

print(f"URL yang perlu di-decode: {len(need_decode)}")

DECODE_WORKERS = 4
DECODE_BATCH = 50

for batch_start in range(0, len(need_decode), DECODE_BATCH):
    batch = need_decode[batch_start:batch_start + DECODE_BATCH]

    with ThreadPoolExecutor(max_workers=DECODE_WORKERS) as executor:
        futures = {executor.submit(safe_decode_url, item): item for item in batch}
        for future in as_completed(futures):
            idx, decoded = future.result()
            df_valid.loc[idx, "Actual_URL"] = decoded

    print(f"Decoded {min(batch_start + DECODE_BATCH, len(need_decode))}/{len(need_decode)} URL")

    if batch_start + DECODE_BATCH < len(need_decode):
        time.sleep(2)

df_valid.to_csv("hasil_scraping_isi_artikel.csv", index=False)
print("Decode selesai, mulai scraping...")


need_scrape = [
    (i, df_valid.loc[i, "Actual_URL"])
    for i in range(len(df_valid))
    if pd.isna(df_valid.loc[i, "Content"]) and pd.notna(df_valid.loc[i, "Actual_URL"])
]

print(f"Artikel yang perlu di-scrape: {len(need_scrape)}")

SCRAPE_WORKERS = 20 
SCRAPE_BATCH = 200

for batch_start in range(0, len(need_scrape), SCRAPE_BATCH):
    batch = need_scrape[batch_start:batch_start + SCRAPE_BATCH]

    with ThreadPoolExecutor(max_workers=SCRAPE_WORKERS) as executor:
        futures = {executor.submit(scrape_content, item): item for item in batch}
        for future in as_completed(futures):
            idx, content = future.result()
            df_valid.loc[idx, "Content"] = content

    df_valid.to_csv("hasil_scraping_isi_artikel.csv", index=False)
    print(f"Scraping {min(batch_start + SCRAPE_BATCH, len(need_scrape))}/{len(need_scrape)} artikel selesai")

print("\nSEMUA ARTIKEL SELESAI")

Total artikel valid: 1144
URL yang perlu di-decode: 1144
Decoded 50/1144 URL


KeyboardInterrupt: 